# Multi-scale figure: from cosmic web to the SMILE galaxy

Produces a 4-panel figure connecting large cosmological scales to a single galaxy and its dust content.

**Panels:**
1. Gas + stars at group/cluster scale (~1 Mpc) — showing the cosmic environment
2. Gas + stars + dust RGB composite of the SMILE galaxy, face-on (~30 kpc)
3. Stellar surface density map (~30 kpc)
4. Dust surface density map (~30 kpc)

A zoom box in panel 1 marks the galaxy region shown in panels 2–4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

import yt
import caesar
import sphviewer as sph
from sphviewer.tools import QuickView, Blend

from simbanator.io.simba import Simba
from simbanator.visualization.rendering import (
    RenderRGB, SingleRender,
    get_normalized_image, apply_dust_screen, find_rot_ax,
)

plt.rcParams.update({
    'savefig.facecolor': 'k',
    'figure.facecolor': 'k',
    'axes.facecolor': 'k',
    'text.color': 'w',
    'axes.edgecolor': 'w',
    'axes.labelcolor': 'w',
    'xtick.color': 'w',
    'ytick.color': 'w',
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'cm',
    'font.size': 14,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

## 1 – Configuration

In [ ]:
# ── Snapshot / galaxy selection ───────────────────────────────────────────────
SNAP = 129          # z ≈ 0.4 for m25n512
SMILE_GAL_ID = 0   # ← set this to the SMILE galaxy index

# Scales (physical kpc half-extents for each panel)
EXT_LARGE = 700    # group environment   ~1.4 Mpc
EXT_GALAXY = 22   # individual galaxy   ~44 kpc

# Render resolution
PIX_LARGE  = 600
PIX_GALAXY = 500

# ── Load simulation ───────────────────────────────────────────────────────────
sb = Simba(box=25)
snapfile = sb.get_sim_file(SNAP)
catfile  = sb.get_caesar_file(SNAP)
z_snap   = sb.get_z_from_snap(SNAP)

print(f"Snapshot {SNAP}: z = {z_snap:.3f}")

obj = caesar.load(catfile)
gal = obj.galaxies[SMILE_GAL_ID]
a   = obj.simulation.scale_factor

center = gal.minpotpos.in_units('kpc').value
L      = gal.rotation['gas_L']
t, p   = find_rot_ax(L, spos='faceon')

print(f"Galaxy {SMILE_GAL_ID}:  "
      f"M★ = {gal.masses['stellar']:.2e} M☉  |  "
      f"M_dust = {gal.masses['dust']:.2e} M☉")
print(f"Center (kpc): {center}")
print(f"Face-on angles: t={t:.1f}°, p={p:.1f}°")

## 2 – Load particles

In [ ]:
import h5py
import gc

# ── Unit factors from snapshot header ────────────────────────────────────────
with h5py.File(snapfile, 'r') as f:
    attrs  = dict(f['Header'].attrs)
    h5_h   = float(attrs['HubbleParam'])
    h5_a   = float(attrs['Time'])                             # scale factor
    # SIMBA snapshots often omit unit attrs; fall back to standard SIMBA values
    u_len  = float(attrs.get('UnitLength_in_cm',  3.085678e21))  # 1 kpc
    u_mass = float(attrs.get('UnitMass_in_g',     1.989e43))     # 1e10 Msun

kpc_in_cm   = 3.085678e21
msun_in_g   = 1.989e33
kpc_factor  = (u_len / kpc_in_cm) * h5_a / h5_h   # code → physical kpc
msun_factor = (u_mass / msun_in_g) / h5_h          # code → Msun

# center is already in physical kpc (from Caesar); convert to code units for masking
center_code = center / kpc_factor
ext_code    = EXT_LARGE / kpc_factor

# ── Large-scale gas: load full array → mask → discard ────────────────────────
print("Loading large-scale gas...")
with h5py.File(snapfile, 'r') as f:
    coords = f['PartType0/Coordinates'][:]
    mask   = np.all(np.abs(coords - center_code) < ext_code, axis=1)
    all_gas_pos  = (coords[mask] * kpc_factor).astype('f4')
    del coords
    masses = f['PartType0/Masses'][:]
    all_gas_mass = (masses[mask] * msun_factor).astype('f4')
    del masses, mask
gc.collect()

# ── Large-scale stars ─────────────────────────────────────────────────────────
print("Loading large-scale stars...")
with h5py.File(snapfile, 'r') as f:
    coords = f['PartType4/Coordinates'][:]
    mask   = np.all(np.abs(coords - center_code) < ext_code, axis=1)
    all_star_pos  = (coords[mask] * kpc_factor).astype('f4')
    del coords
    masses = f['PartType4/Masses'][:]
    all_star_mass = (masses[mask] * msun_factor).astype('f4')
    del masses, mask
gc.collect()

# ── Galaxy-member particles: read only the indexed rows ───────────────────────
glist = np.sort(gal.glist)
slist = np.sort(gal.slist)

print("Loading galaxy particles...")
with h5py.File(snapfile, 'r') as f:
    gal_gas_pos   = (f['PartType0/Coordinates'][glist] * kpc_factor).astype('f4')
    gal_gas_mass  = (f['PartType0/Masses'][glist]      * msun_factor).astype('f4')
    gal_dust_mass = (f['PartType0/Dust_Masses'][glist] * msun_factor).astype('f4')
    gal_star_pos  = (f['PartType4/Coordinates'][slist] * kpc_factor).astype('f4')
    gal_star_mass = (f['PartType4/Masses'][slist]      * msun_factor).astype('f4')

gal_dust_pos = gal_gas_pos   # dust lives on gas particles — alias, no copy needed
gc.collect()

print(f"Large-scale: {len(all_gas_pos):,} gas | {len(all_star_pos):,} stars")
print(f"Galaxy:      {len(gal_gas_pos):,} gas | {len(gal_star_pos):,} stars")

## 3 – Render the four panels

In [ ]:
# ── Helper: make an sphviewer Camera pointed face-on at center ───────────────
def make_camera(center, extent, xsize, ysize, t, p):
    return sph.Camera(
        x=center[0], y=center[1], z=center[2],
        r='infinity', t=t, p=p, roll=0,
        extent=[-extent, extent, -extent, extent],
        xsize=xsize, ysize=ysize,
    )

def render(particles, camera):
    S = sph.Scene(particles, Camera=camera)
    R = sph.Render(S)
    return R.get_image(), R.get_extent()


# ── Panel A: large-scale environment ─────────────────────────────────────────
# Top-down view: no rotation (t=0, p=90 = looking down z-axis)
print("Rendering large-scale panel...")
cam_large = make_camera(center, EXT_LARGE, PIX_LARGE, PIX_LARGE, t=0, p=90)

img_gas_large,  ext_large = render(sph.Particles(all_gas_pos,  all_gas_mass),  cam_large)
img_star_large, _         = render(sph.Particles(all_star_pos, all_star_mass), cam_large)

gas_norm_large  = get_normalized_image(img_gas_large,  vmin=1, vmax=99.5, mode='log')
star_norm_large = get_normalized_image(img_star_large, vmin=1, vmax=99.5, mode='log')

blend_large = Blend.Blend(cm.Greys_r(star_norm_large), cm.magma(gas_norm_large))
rgb_large   = blend_large.Screen()
print("  done.")

# ── Panel B: galaxy RGB (gas + stars + dust attenuation) ─────────────────────
print("Rendering galaxy RGB panel...")
cam_gal = make_camera(center, EXT_GALAXY, PIX_GALAXY, PIX_GALAXY, t, p)

img_gas_gal,  ext_gal = render(sph.Particles(gal_gas_pos,  gal_gas_mass),  cam_gal)
img_star_gal, _       = render(sph.Particles(gal_star_pos, gal_star_mass), cam_gal)
img_dust_gal, _       = render(sph.Particles(gal_dust_pos, gal_dust_mass), cam_gal)

gas_norm  = get_normalized_image(img_gas_gal,  vmin=1, vmax=99.5, mode='log')
star_norm = get_normalized_image(img_star_gal, vmin=1, vmax=99.5, mode='log')
dust_norm = get_normalized_image(img_dust_gal, vmin=1, vmax=99,   mode='log')

gas_att  = apply_dust_screen(gas_norm,  dust_norm, tau_max=1.5)
star_att = apply_dust_screen(star_norm, dust_norm, tau_max=1.5)

blend_gal = Blend.Blend(cm.Greys_r(star_att), cm.afmhot(gas_att))
rgb_galaxy = blend_gal.Screen()
print("  done.")

# ── Panel C: stellar surface density ─────────────────────────────────────────
print("Rendering stellar map...")
img_stars_map = get_normalized_image(img_star_gal, vmin=5, vmax=99.5, mode='log')
print("  done.")

# ── Panel D: dust surface density ────────────────────────────────────────────
print("Rendering dust map...")
img_dust_map = get_normalized_image(img_dust_gal, vmin=5, vmax=99.5, mode='log')
print("  done.")

## 4 – Compose the figure

In [ ]:
# ── Layout ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 6), facecolor='k')
gs = gridspec.GridSpec(
    1, 4,
    figure=fig,
    wspace=0.04,
    left=0.02, right=0.98,
    top=0.97, bottom=0.05,
    width_ratios=[1.35, 1, 1, 1],  # large panel slightly wider
)

ax_large = fig.add_subplot(gs[0])
ax_rgb   = fig.add_subplot(gs[1])
ax_star  = fig.add_subplot(gs[2])
ax_dust  = fig.add_subplot(gs[3])

for ax in [ax_large, ax_rgb, ax_star, ax_dust]:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('w')
        spine.set_linewidth(1.2)

# ── Panel A: large-scale ─────────────────────────────────────────────────────
ax_large.imshow(rgb_large, extent=ext_large, origin='lower')

# Zoom box: indicate where the galaxy panel sits
box_half = EXT_GALAXY
rect = Rectangle(
    (-box_half, -box_half), 2*box_half, 2*box_half,
    linewidth=1.5, edgecolor='cyan', facecolor='none', linestyle='--',
)
ax_large.add_patch(rect)

ax_large.set_xlabel(r'$x\;[\mathrm{kpc}]$', color='w', fontsize=13)
ax_large.set_ylabel(r'$y\;[\mathrm{kpc}]$', color='w', fontsize=13)
ax_large.tick_params(labelcolor='w', labelsize=11)
ax_large.set_xticks(np.linspace(-EXT_LARGE, EXT_LARGE, 5).astype(int))
ax_large.set_yticks(np.linspace(-EXT_LARGE, EXT_LARGE, 5).astype(int))

# Scale bar: 200 kpc
scale_kpc  = 200
ax_large.plot(
    [ext_large[0] + 60, ext_large[0] + 60 + scale_kpc],
    [ext_large[2] + 60, ext_large[2] + 60],
    lw=3, color='w',
)
ax_large.text(
    ext_large[0] + 60 + scale_kpc/2, ext_large[2] + 130,
    f'{scale_kpc} kpc', color='w', ha='center', fontsize=12,
)

ax_large.set_title(r'Group environment  ($z={:.2f}$)'.format(z_snap),
                    color='w', fontsize=14, pad=5)

# ── Panel B: galaxy RGB ───────────────────────────────────────────────────────
ax_rgb.imshow(rgb_galaxy, extent=ext_gal, origin='lower')
ax_rgb.set_title('SMILE galaxy\n(gas + stars + dust)', color='w', fontsize=13, pad=5)

# Scale bar: 5 kpc
scale_kpc_gal = 5
ax_rgb.plot(
    [ext_gal[0] + 2, ext_gal[0] + 2 + scale_kpc_gal],
    [ext_gal[2] + 2, ext_gal[2] + 2],
    lw=3, color='w',
)
ax_rgb.text(
    ext_gal[0] + 2 + scale_kpc_gal/2, ext_gal[2] + 3.5,
    f'{scale_kpc_gal} kpc', color='w', ha='center', fontsize=12,
)

# ── Panel C: stellar surface density ─────────────────────────────────────────
im_star = ax_star.imshow(
    img_stars_map, extent=ext_gal, origin='lower',
    cmap='YlOrRd', vmin=0, vmax=1,
)
ax_star.set_title(r'$\Sigma_\star$  [arb.]', color='w', fontsize=13, pad=5)

ax_star.plot(
    [ext_gal[0] + 2, ext_gal[0] + 2 + scale_kpc_gal],
    [ext_gal[2] + 2, ext_gal[2] + 2],
    lw=3, color='w',
)
ax_star.text(
    ext_gal[0] + 2 + scale_kpc_gal/2, ext_gal[2] + 3.5,
    f'{scale_kpc_gal} kpc', color='w', ha='center', fontsize=12,
)

# ── Panel D: dust surface density ────────────────────────────────────────────
im_dust = ax_dust.imshow(
    img_dust_map, extent=ext_gal, origin='lower',
    cmap='inferno', vmin=0, vmax=1,
)
ax_dust.set_title(r'$\Sigma_\mathrm{dust}$  [arb.]', color='w', fontsize=13, pad=5)

ax_dust.plot(
    [ext_gal[0] + 2, ext_gal[0] + 2 + scale_kpc_gal],
    [ext_gal[2] + 2, ext_gal[2] + 2],
    lw=3, color='w',
)
ax_dust.text(
    ext_gal[0] + 2 + scale_kpc_gal/2, ext_gal[2] + 3.5,
    f'{scale_kpc_gal} kpc', color='w', ha='center', fontsize=12,
)

# ── Zoom arrow from panel A to panel B ───────────────────────────────────────
# Draw a connection in figure coordinates from the cyan box in ax_large
# to the left edge of ax_rgb.
# We compute the figure position of the top-right corner of the zoom box.
def ax_to_fig(ax, x_data, y_data):
    """Convert data coords → figure coords."""
    disp = ax.transData.transform([x_data, y_data])
    return fig.transFigure.inverted().transform(disp)

x0_box_fig = ax_to_fig(ax_large,  box_half, 0)[0]
y0_box_fig = ax_to_fig(ax_large, 0, 0)[1]
x1_rgb_fig = ax_to_fig(ax_rgb, ext_gal[0], 0)[0]

con_top = plt.annotate(
    '', xy=(x1_rgb_fig, 0.97), xycoords='figure fraction',
    xytext=(x0_box_fig, 0.97), textcoords='figure fraction',
    arrowprops=dict(arrowstyle='->', color='cyan', lw=1.5),
    annotation_clip=False,
)
con_bot = plt.annotate(
    '', xy=(x1_rgb_fig, 0.05), xycoords='figure fraction',
    xytext=(x0_box_fig, 0.05), textcoords='figure fraction',
    arrowprops=dict(arrowstyle='->', color='cyan', lw=1.5),
    annotation_clip=False,
)

# ── Legend text for large panel ───────────────────────────────────────────────
legend_handles = [
    Line2D([0], [0], color=cm.magma(0.7), linewidth=4, label='Gas'),
    Line2D([0], [0], color='0.8',          linewidth=4, label='Stars'),
]
ax_large.legend(handles=legend_handles, loc='upper right',
                fontsize=11, framealpha=0.35, facecolor='k', labelcolor='w')

# ── Save ─────────────────────────────────────────────────────────────────────
import os
out_dir = os.path.join(os.getcwd(), 'output', 'figures')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f'multiscale_snap{SNAP}_gal{SMILE_GAL_ID}.pdf')
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='k')
print(f"Figure saved → {out_path}")
plt.show()

## 5 – Optional: add SED panel

If you have a Powderday `.rtout.sed` output for this galaxy, run the cell below to produce a 5-panel figure with the SED on the right.

In [ ]:
# ── Requires: hyperion, Powderday output ─────────────────────────────────────
from hyperion.model import ModelOutput
from astropy.cosmology import Planck13
from astropy import units as u, constants

SED_SNAP = SNAP       # snapshot for which SED exists
SED_GAL  = SMILE_GAL_ID

# Paths — adjust to match your Powderday output layout
sed_run_dir = f'/mnt/home/glorenzon/simbanator/output/hdf5/powderday_sed_out'
sed_path     = (f'{sed_run_dir}/snap_{SED_SNAP:03d}/gal_{SED_GAL:06d}/'
                f'snap{SED_SNAP:03d}.galaxy{SED_GAL:06d}.rtout.sed')

# Also load no-dust SED if available
sed_nodust_dir  = sed_run_dir.replace('powderday_sed_out', 'powderday_sed_nodust')
sed_nodust_path = (f'{sed_nodust_dir}/snap_{SED_SNAP:03d}/gal_{SED_GAL:06d}/'
                   f'snap{SED_SNAP:03d}.galaxy{SED_GAL:06d}.rtout.sed')

import os
print("SED with dust:",    os.path.exists(sed_path))
print("SED without dust:", os.path.exists(sed_nodust_path))

In [ ]:
def load_sed(path, z):
    """Load a Powderday SED; return (wavelength_um, L_lambda array [erg/s/um])."""
    m = ModelOutput(path)
    wav_um, flux_erg_s = m.get_sed(inclination='all', aperture=-1)
    wav_um  = np.asarray(wav_um) * (1 + z)   # redshift
    flux    = np.asarray(flux_erg_s)           # shape (n_inclinations, n_wav)
    # Convert to nu*F_nu = lambda*F_lambda (still erg/s, just redistributed)
    # Keep as L_lambda [erg/s/μm] = flux / wav_um  for a tidy loglog plot
    return wav_um, flux

wav, sed_dust    = load_sed(sed_path,        z_snap)
try:
    _, sed_nodust = load_sed(sed_nodust_path, z_snap)
    has_nodust = True
except Exception:
    has_nodust = False
    print("No-dust SED not found — showing dust SED only.")

# ── 5-panel figure with SED ───────────────────────────────────────────────────
fig5 = plt.figure(figsize=(28, 6), facecolor='k')
gs5  = gridspec.GridSpec(
    1, 5,
    figure=fig5,
    wspace=0.07,
    left=0.02, right=0.98,
    top=0.95, bottom=0.12,
    width_ratios=[1.35, 1, 1, 1, 1.4],
)

axA = fig5.add_subplot(gs5[0])
axB = fig5.add_subplot(gs5[1])
axC = fig5.add_subplot(gs5[2])
axD = fig5.add_subplot(gs5[3])
axE = fig5.add_subplot(gs5[4])

# Panels A–D: same images as before
for ax in [axA, axB, axC, axD]:
    ax.set_xticks([]); ax.set_yticks([])

axA.imshow(rgb_large,  extent=ext_large, origin='lower')
rect2 = Rectangle((-box_half, -box_half), 2*box_half, 2*box_half,
                   lw=1.5, ec='cyan', fc='none', ls='--')
axA.add_patch(rect2)
axA.set_title(r'Group environment  ($z={:.2f}$)'.format(z_snap), color='w', fontsize=12)

axB.imshow(rgb_galaxy, extent=ext_gal, origin='lower')
axB.set_title('SMILE galaxy (RGB)', color='w', fontsize=12)

axC.imshow(img_stars_map, extent=ext_gal, origin='lower', cmap='YlOrRd', vmin=0, vmax=1)
axC.set_title(r'$\Sigma_\star$', color='w', fontsize=12)

axD.imshow(img_dust_map, extent=ext_gal, origin='lower', cmap='inferno', vmin=0, vmax=1)
axD.set_title(r'$\Sigma_\mathrm{dust}$', color='w', fontsize=12)

# Panel E: SED
axE.set_facecolor('k')
for sp in axE.spines.values(): sp.set_edgecolor('0.5')

# Average over inclinations; plot individual inclinations lightly
colors_inc = plt.cm.Blues(np.linspace(0.35, 0.9, sed_dust.shape[0]))
for i, (row, col) in enumerate(zip(sed_dust, colors_inc)):
    axE.loglog(wav, row, color=col, lw=0.8, alpha=0.6)
axE.loglog(wav, sed_dust.mean(axis=0),
           color='deepskyblue', lw=2.5, label='With dust')

if has_nodust:
    axE.loglog(wav, sed_nodust.mean(axis=0),
               color='orange', lw=2.5, ls='--', label='No dust')

# Mark photometric bands
band_wav = {'FUV': 0.15, 'r': 0.62, 'K': 2.2, 'IRAC 3.6': 3.6,
            'MIPS 24': 24, 'PACS 100': 100, 'SPIRE 250': 250, '850': 850}
for label, wv in band_wav.items():
    axE.axvline(wv, color='0.35', lw=0.7, ls=':')
    axE.text(wv, axE.get_ylim()[1] if axE.get_ylim()[1] > 0 else 1e35,
             label, rotation=90, va='top', ha='center',
             fontsize=8, color='0.6')

axE.set_xlabel(r'$\lambda\;[\mu\mathrm{m}]$', color='w', fontsize=13)
axE.set_ylabel(r'$L_\lambda\;[\mathrm{erg\,s^{-1}}]$', color='w', fontsize=13)
axE.set_xlim(0.08, 2000)
axE.tick_params(colors='w', labelsize=11)
axE.legend(fontsize=11, framealpha=0.3, facecolor='k', labelcolor='w')
axE.set_title('SED — dust vs. no dust', color='w', fontsize=12)

out5 = os.path.join(out_dir, f'multiscale_sed_snap{SNAP}_gal{SMILE_GAL_ID}.pdf')
fig5.savefig(out5, dpi=200, bbox_inches='tight', facecolor='k')
print(f"5-panel figure saved → {out5}")
plt.show()